# Phase 5: bug report to past fix commits

Upload **phase5_bugfix_v2_source.zip** to My Drive and open this notebook on an **A100 40 GB**,
and run the cells in order. Bundled frozen datasets need no GitHub key in Colab.
The v2 dataset is complete: no live GitHub selection occurs in this notebook.
No paid LLM key is needed. The separate ZIP keeps notebook cells short; there is no base64 archive.

V2 preserves the 329 v1 queries, gold labels and connected-group splits, and corrects the
commit scrub to remove reference phrases while keeping adjacent descriptions. Inputs and
protocol were re-frozen before v2 retrieval. This is a correction, not a fresh holdout.
It compares full-query BM25, a 2000-character BM25 sensitivity check, dense, hybrid,
4B reranking and 8B reranking at fixed depth 50.
Both rerankers use the bug-to-fix instruction. No weight tuning or test-set selection occurs.

Issue text is used as the query only. Commit documents contain message, files, AST symbols and
800 characters of scrubbed diff. Links are proxy labels; not every closing issue is a bug and
other valid fixes may be unlabelled. Requests queries may overlap the earlier T3 benchmark.
This measures finding an issue's own fix, not similar historical fixes for new bugs.
File Hit@5 is a secondary same-path proxy: the exact fix itself qualifies, and sharing a path
does not prove semantic relevance. Every gold qualifies from the message heuristic alone;
closing links do not determine evaluation pool membership. The broad fix heuristic remains
a limitation.
BM25 sees full queries; the sensitivity row uses 2000 characters. Embedding and reranker
truncation still have different token budgets. Query-length strata are reported separately.
The fixed 8B rule retains pooled MRR gain >=0.02 and a positive paired CI lower bound;
repo/link-path guards apply only with >=30 connected test groups. FastAPI is descriptive,
with no safety conclusion for its 14 test queries. Failure or inconclusive data is valid.

The explicit model-download cell fetches roughly 25 GB of weights. This GPU run's duration
has not been measured.
Preparation is backed up to Drive; keep returned results intact. Frozen source records retain
public historical text (including email trailers); exported metadata.db redacts email-like strings.

The result ZIP contains raw per-question results, full candidate/pair scores, source and input
hashes, a reproducible analysis, a portable commit index, metadata.db and a real CLI citation.
This tests retrieval, not generated answers, scale limits or deployment latency.

In [ ]:
SOURCE_ZIP = "/content/drive/MyDrive/phase5_bugfix_v2_source.zip"

In [ ]:
# Load the separate source ZIP that you uploaded to Google Drive.
import hashlib, pathlib, zipfile
from google.colab import drive

drive.mount("/content/drive")
source_zip = pathlib.Path(SOURCE_ZIP)
if not source_zip.is_file():
    raise FileNotFoundError(
        f"Upload phase5_bugfix_v2_source.zip to My Drive, or update SOURCE_ZIP. Missing: {source_zip}"
    )
expected_sha256 = "418469c87eb4b56d49c7864c7f0f610ef4978c2910973ae1350f1b6df0646a3b"
if hashlib.sha256(source_zip.read_bytes()).hexdigest() != expected_sha256:
    raise ValueError(
        "Source ZIP does not match this notebook. Download the matching ZIP and notebook again."
    )
PAYLOAD = pathlib.Path("/content/phase5_payload")
with zipfile.ZipFile(source_zip) as package:
    package.extractall(PAYLOAD)
print("Loaded project code and question datasets from", source_zip.name)

In [ ]:
import importlib.util, subprocess, sys, os, pathlib, json
subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv'], check=True)
if importlib.util.find_spec('uv') is None:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'uv'], check=True)
subprocess.run([sys.executable, '-m', 'uv', 'pip', 'install', '--system',
                '--python', sys.executable, '-q', 'chromadb==1.5.9', 'fastapi==0.162.0', 'gitpython==3.2.1', 'langchain-chroma==1.1.0', 'langchain-core==1.6.13', 'langchain-text-splitters==1.1.3', 'langgraph==1.2.19', 'pydantic==2.14.1', 'pydantic-settings==2.16.1', 'rank-bm25==0.2.2', 'rich==15.0.0', 'sentence-transformers==5.7.0', 'transformers==4.57.6', 'tree-sitter==0.26.0', 'tree-sitter-language-pack==1.22.1', 'typer==0.27.3', 'uvicorn==0.54.0'], check=True)
from tree_sitter_language_pack import download
download(['python', 'javascript', 'typescript', 'tsx', 'java', 'go'])
ENV = {**os.environ, 'PYTHONPATH': os.pathsep.join([str(PAYLOAD), str(PAYLOAD / 'src')]),
       'PYTHONUNBUFFERED': '1', 'TOKENIZERS_PARALLELISM': 'false'}
RESULTS = pathlib.Path('/content/phase5_bugfix_v2_results')
RESULTS.mkdir(exist_ok=True)
PROTOCOL = json.loads((PAYLOAD / 'eval/datasets/phase5_bugfix_v2/protocol.json').read_text())
def execute(script, *args, env=ENV):
    subprocess.run([sys.executable, str(PAYLOAD / script), *map(str, args)],
                   env=env, check=True)

In [ ]:
# V2 must use these exact corrected, frozen inputs; no live re-selection.
import shutil
sys.path[:0] = [str(PAYLOAD), str(PAYLOAD / 'src')]
from eval.prepare_bugfix_eval import verify_corpus
for repo in PROTOCOL['repos']:
    stem = repo.replace('/', '__')
    bundled = PAYLOAD / 'eval/datasets/phase5_bugfix_v2/repos' / stem
    dataset = RESULTS / 'datasets' / stem
    if not (bundled / 'freeze.json').exists():
        raise FileNotFoundError('Matching v2 source ZIP must include frozen datasets')
    verify_corpus(bundled)
    if not dataset.exists():
        shutil.copytree(bundled, dataset)
    verify_corpus(dataset)
    print(repo, 'v2 corpus verified before retrieval', flush=True)
backup = pathlib.Path('/content/drive/MyDrive/phase5_bugfix_v2_preparation.zip')
shutil.make_archive(str(backup.with_suffix('')), 'zip', RESULTS / 'datasets')
print('Preparation backup:', backup)

In [ ]:
# Explicit download cell: no LLM service or Anthropic key is used.
from huggingface_hub import snapshot_download
for model in ['Qwen/Qwen3-Embedding-0.6B', 'Qwen/Qwen3-Reranker-4B', 'Qwen/Qwen3-Reranker-8B']:
    print('Downloading', model, flush=True)
    snapshot_download(model)

In [ ]:
# One process per method: the 4B and 8B never occupy GPU memory together.
import shutil
shutil.copy2(source_zip, RESULTS / 'source.zip')
for repo in PROTOCOL['repos']:
    stem = repo.replace('/', '__')
    for method in PROTOCOL['retrieval']['methods']:
        output = RESULTS / 'runs' / stem / method
        if (output / 'run.json').exists():
            print('Already complete:', repo, method)
            continue
        if output.exists():
            raise FileExistsError(f'Incomplete: {output}; preserve it and use a fresh runtime')
        execute('eval/run_bugfix_eval.py', '--dataset', RESULTS / 'datasets' / stem,
                '--method', method, '--output', output, '--device', 'cuda')
execute('eval/analyze_bugfix_run.py', RESULTS)
execute('eval/export_bugfix_index.py', RESULTS)
print((RESULTS / 'aggregate/summary.md').read_text())

In [ ]:
# Ship evidence, metadata and a vector ZIP; skip Chroma runtime files.
import zipfile
archive = pathlib.Path('/content/phase5_bugfix_v2_results.zip')
with zipfile.ZipFile(archive, 'w', zipfile.ZIP_DEFLATED) as bundle:
    for path in sorted(RESULTS.rglob('*')):
        rel = path.relative_to(RESULTS)
        if not path.is_file() or 'index' in rel.parts or 'production_index' in rel.parts:
            continue
        bundle.write(path, rel.as_posix())
    bundle.write(RESULTS / 'production_index/metadata.db', 'metadata.db')
shutil.copy2(archive, '/content/drive/MyDrive/phase5_bugfix_v2_results.zip')
from google.colab import files
files.download(str(archive))